2026-11-16 학습

코드 작성 **Gemini-4-Hydrogen-High**

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

In [2]:
# 1. 단어 사전 및 입력 문장 준비 ("영화 정말 재밌다")
vocab = {"영화": 0, "정말": 1, "재밌다": 2}
words = ["영화", "정말", "재밌다"]
input_ids = torch.tensor([[vocab[w] for w in words]])  # shape: (배치=1, 문장길이=3)

In [ ]:
class IntuitiveRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # RNN 핵심 가중치 2개
        self.W_x = nn.Linear(embed_dim, hidden_dim)              # 현재 단어(x_t)용
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False) # 이전 기억(h_{t-1})용
        
        # 최종 분류기
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        batch_size, seq_len = x.shape
        embeds = self.embedding(x)  # shape: (1, 3, embed_dim)

        # 초기 기억(h_{-1})은 0으로 채워진 빈 벡터로 시작
        h = torch.zeros(batch_size, self.hidden_dim)

        # ★ RNN의 본질: 문장 길이(3)만큼 스텝을 밟으며 기억(h)을 갱신
        for t in range(seq_len):
            x_t = embeds[:, t, :]  # t번째 단어 벡터 꺼내기
            
            # h_t = tanh(W_x * x_t + W_h * h_{t-1} + b)
            h = torch.tanh(self.W_x(x_t) + self.W_h(h))
            
            print(f"Step {t} ('{words[t]}' 입력 후) -> 은닉 상태 h_{t}: {h.detach().numpy().round(3)}")

        # 모든 단어를 다 읽은 '마지막 기억(h_2)'만 출력층에 통과시킴
        logits = self.fc(h)
        probs = torch.softmax(logits, dim=-1)
        return probs

In [4]:
# 모델 생성 (단어수 3, 임베딩 크기 4, 기억 크기 3, 클래스 2개[긍정/부정])
model = IntuitiveRNN(vocab_size=3, embed_dim=4, hidden_dim=3, num_classes=2)
output = model(input_ids)

print(f"\n최종 출력 확률 [긍정, 부정]: {output.detach().numpy().round(3)}")

Step 0 ('영화' 입력 후) -> 은닉 상태 h_0: [[-0.185  0.349  0.225]]
Step 1 ('정말' 입력 후) -> 은닉 상태 h_1: [[ 0.46   0.958 -0.643]]
Step 2 ('재밌다' 입력 후) -> 은닉 상태 h_2: [[-0.309  0.524  0.504]]

최종 출력 확률 [긍정, 부정]: [[0.329 0.671]]


## RNN은 Vector Embedding이 필수인가?

<br><br>

아닙니다. RNN 자체에는 **벡터 임베딩(`nn.Embedding`)이 필수가 아닙니다.** RNN이 요구하는 유일한 조건은 매 스텝 들어가는 입력($x_t$)이 **'숫자 벡터'** 형태이기만 하면 된다는 것입니다.

임베딩이 필요한지 여부는 RNN의 구조 때문이 아니라, 다루려는 데이터가 원래 숫자인지(연속형), 글자·기호인지(이산형)에 따라 갈립니다.

---

### 1. 이미 숫자인 데이터: 임베딩 불필요

주가, 기온, 심전도, 음성 파형 같은 **시계열 데이터**는 각 스텝의 값이 이미 의미 있는 숫자입니다.

* **예시 (일별 주가 예측):** 매일의 `[시가, 종가, 거래량]`을 묶으면 그 자체로 3차원 숫자 벡터가 됩니다.
* Day 0 입력 $x_0$: `[70000, 71200, 1500000]`
* Day 1 입력 $x_1$: `[71500, 70800, 1200000]`


* 이 경우 CNN에 이미지 픽셀(RGB 숫자)을 그대로 넣는 것처럼, **`nn.Embedding` 없이 숫자 벡터를 `nn.RNN`에 바로 집어넣습니다.**

---

### 2. 단어·문자 같은 기호 데이터: 숫자 변환 필수 (임베딩 외 대안 있음)

`"영화"`, `"정말"` 같은 텍스트는 글자이므로 가중치 행렬($W_x x_t$)과 곱셈을 할 수 없습니다. 따라서 **어떤 방식으로든 숫자 벡터로 바꾸는 과정**은 반드시 거쳐야 합니다.

이때 `nn.Embedding`을 쓰지 않고 원-핫 인코딩(One-Hot Encoding)만 써도 RNN은 정상적으로 돌아갑니다.

* `"영화"` $\rightarrow$ `[1, 0, 0]`
* `"정말"` $\rightarrow$ `[0, 1, 0]`
* `"재밌다"` $\rightarrow$ `[0, 0, 1]`

그럼에도 자연어 처리에서 벡터 임베딩을 거의 필수처럼 쓰는 이유는 **원-핫 인코딩의 두 가지 치명적 한계** 때문입니다.

| 구분 | 원-핫 인코딩 (One-Hot) | 벡터 임베딩 (`nn.Embedding`) |
| --- | --- | --- |
| **벡터 크기** | 단어 사전 크기만큼 커짐 (단어가 5만 개면 5만 차원) | 지정한 크기로 압축 (보통 128~768차원) |
| **벡터 형태** | `[0, 0, ..., 1, ..., 0]` (대부분 0인 희소 벡터) | `[0.24, -0.81, 0.05, ...]` (값이 꽉 찬 밀집 벡터) |
| **의미 반영** | 모든 단어 간 거리가 동일 (`"영화"`와 `"드라마"`의 관계 파악 불가) | 비슷한 맥락의 단어끼리 비슷한 숫자 좌표를 가짐 |

---

### CNN과 비교하면 명확해지는 이유

CNN에서 이미지 픽셀은 임베딩 없이 바로 들어갑니다. 픽셀 값 `254`와 `255`는 "비슷하게 밝은 색"이라는 연속적인 수치 의미를 이미 갖고 있기 때문입니다.

반면 단어에 단순히 번호를 매기면 `"영화"=0`, `"드라마"=1`, `"자동차"=2`가 되는데, 컴퓨터는 이를 보고 "자동차가 영화의 2배"라고 오해할 수 있습니다. 그래서 언어 모델에서는 단어 번호를 그대로 쓰지 않고, **단어마다 고유한 의미 좌표를 학습시켜 주는 `nn.Embedding` 층을 RNN 앞단에 끼워 넣는 것**입니다.